In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
load_dotenv()

True

In [4]:
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    temperature=0.7,
   
)

response = llm.invoke("Hello")
print(response.content)

Hello! How can I assist you today?


In [5]:
llm_google = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",   
    temperature=0.7
)   
response_google = llm_google.invoke("Hello")
print(response_google.content)

Hello! How can I help you today?


In [6]:
class JokeState(TypedDict):    #here we define the structure of the state that will be used in the workflow

    topic: str
    joke: str
    explanation: str

CHATGPT

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

GOOGLE

In [7]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm_google.invoke(prompt).content

    return {'joke': response}

CHATGPT

In [8]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

GOOGLE

In [9]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm_google.invoke(prompt).content

    return {'explanation': response}

In [10]:
graph = StateGraph(JokeState)                                   #here we create a state graph that will define the workflow of our application. The graph will have nodes that represent different states and edges that represent the transitions between those states.

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()                                  #here we create a checkpointer that will be used to save the state of the workflow. The checkpointer will save the state of the workflow in memory.

workflow = graph.compile(checkpointer=checkpointer)             #here we compile the state graph into a workflow that can be executed. The workflow will be executed by calling the run method on the workflow object.

In [12]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'Cricket'}, config=config1)

{'topic': 'Cricket',
 'joke': 'Why did the cricket team bring a ladder to the game?\n\nBecause they heard the scores were going to be high!',
 'explanation': 'This joke is a classic play on words, specifically with the word "high."\n\nHere\'s the breakdown:\n\n1.  **"High scores" in cricket (and most sports):** This phrase means that a team has scored a lot of runs – a large, impressive number. It\'s a numerical quantity. For example, a score of 300 runs in a one-day match would be considered a "high score."\n\n2.  **"High" in relation to a ladder:** A ladder is used to reach things that are physically "high up" – elevated, far off the ground, or out of reach.\n\nThe humor comes from the punchline deliberately misinterpreting the common sports phrase. The cricket team isn\'t expecting the *physical* scores to be high (as if they were written on a scoreboard that was literally sky-high and needed a ladder to read). Instead, they are expecting the *numerical* scores to be high – meaning 

In [13]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team bring a ladder to the game?\n\nBecause they heard the scores were going to be high!', 'explanation': 'This joke is a classic play on words, specifically with the word "high."\n\nHere\'s the breakdown:\n\n1.  **"High scores" in cricket (and most sports):** This phrase means that a team has scored a lot of runs – a large, impressive number. It\'s a numerical quantity. For example, a score of 300 runs in a one-day match would be considered a "high score."\n\n2.  **"High" in relation to a ladder:** A ladder is used to reach things that are physically "high up" – elevated, far off the ground, or out of reach.\n\nThe humor comes from the punchline deliberately misinterpreting the common sports phrase. The cricket team isn\'t expecting the *physical* scores to be high (as if they were written on a scoreboard that was literally sky-high and needed a ladder to read). Instead, they are expecting the *numerical* scores to

In [14]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team bring a ladder to the game?\n\nBecause they heard the scores were going to be high!', 'explanation': 'This joke is a classic play on words, specifically with the word "high."\n\nHere\'s the breakdown:\n\n1.  **"High scores" in cricket (and most sports):** This phrase means that a team has scored a lot of runs – a large, impressive number. It\'s a numerical quantity. For example, a score of 300 runs in a one-day match would be considered a "high score."\n\n2.  **"High" in relation to a ladder:** A ladder is used to reach things that are physically "high up" – elevated, far off the ground, or out of reach.\n\nThe humor comes from the punchline deliberately misinterpreting the common sports phrase. The cricket team isn\'t expecting the *physical* scores to be high (as if they were written on a scoreboard that was literally sky-high and needed a ladder to read). Instead, they are expecting the *numerical* scores t

In [15]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'football'}, config=config2)

{'topic': 'football',
 'joke': 'Why did the football player bring a pencil to the game?\n\nSo he could **draw** a foul!',
 'explanation': 'This joke is a pun that plays on the two different meanings of the word "**draw**."\n\nHere\'s the breakdown:\n\n1.  **"Draw" (meaning 1 - with a pencil):** The first part of the joke ("Why did the football player bring a pencil to the game?") makes you think of the most common meaning of "draw" when associated with a pencil: to create an image or picture. You imagine the player literally sketching something.\n\n2.  **"Draw a foul" (meaning 2 - in sports):** In sports, particularly football (soccer or American football), "to draw a foul" is an idiom that means to perform an action that causes an opposing player to commit a foul against you, or to make the referee call a foul. Players might strategically position themselves or make certain moves to provoke an opponent into committing an infraction, hoping to get a free kick, penalty, or stop play.\n\

In [16]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'football', 'joke': 'Why did the football player bring a pencil to the game?\n\nSo he could **draw** a foul!', 'explanation': 'This joke is a pun that plays on the two different meanings of the word "**draw**."\n\nHere\'s the breakdown:\n\n1.  **"Draw" (meaning 1 - with a pencil):** The first part of the joke ("Why did the football player bring a pencil to the game?") makes you think of the most common meaning of "draw" when associated with a pencil: to create an image or picture. You imagine the player literally sketching something.\n\n2.  **"Draw a foul" (meaning 2 - in sports):** In sports, particularly football (soccer or American football), "to draw a foul" is an idiom that means to perform an action that causes an opposing player to commit a foul against you, or to make the referee call a foul. Players might strategically position themselves or make certain moves to provoke an opponent into committing an infraction, hoping to get a free kick, penalt

In [17]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'football', 'joke': 'Why did the football player bring a pencil to the game?\n\nSo he could **draw** a foul!', 'explanation': 'This joke is a pun that plays on the two different meanings of the word "**draw**."\n\nHere\'s the breakdown:\n\n1.  **"Draw" (meaning 1 - with a pencil):** The first part of the joke ("Why did the football player bring a pencil to the game?") makes you think of the most common meaning of "draw" when associated with a pencil: to create an image or picture. You imagine the player literally sketching something.\n\n2.  **"Draw a foul" (meaning 2 - in sports):** In sports, particularly football (soccer or American football), "to draw a foul" is an idiom that means to perform an action that causes an opposing player to commit a foul against you, or to make the referee call a foul. Players might strategically position themselves or make certain moves to provoke an opponent into committing an infraction, hoping to get a free kick, penal

### Time Travel

In [29]:
workflow.get_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f177cd5-6d1d-667a-8000-b94df2510676"}})

StateSnapshot(values={'topic': 'football'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f177cd5-6d1d-667a-8000-b94df2510676'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-07-04T17:25:23.378956+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f177cd5-6d12-6292-bfff-c969812b65b7'}}, tasks=(PregelTask(id='c2bfb9eb-5eac-c3c4-910d-681b11d85586', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the football player bring a pencil to the game?\n\nSo he could **draw** a foul!'}),), interrupts=())

In [30]:
workflow.invoke(None, {"configurable": {"thread_id": "2", "checkpoint_id": "1f177cd5-6d12-6292-bfff-c969812b65b7"}})

{'topic': 'football',
 'joke': 'Why did the football player bring a pencil to the game?\n\nBecause he wanted to *draw* a foul!',
 'explanation': 'This joke is a pun that plays on the two different meanings of the word "draw":\n\n1.  **"Draw" (with a pencil):** This is the literal meaning implied by the football player bringing a pencil to the game. You use a pencil to create a picture or sketch. The setup makes you think of artistic drawing.\n\n2.  **"Draw a foul" (in sports):** In football (or soccer, basketball, etc.), to "draw a foul" means to provoke or cause an opposing player to commit an infraction against you. It\'s about getting the referee to call a penalty on the other team.\n\nThe humor comes from the **unexpected switch** in the meaning of "draw." The setup leads you to think of the artistic meaning, but the punchline uses the sports-specific meaning, creating a silly and clever play on words.'}

In [31]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'football', 'joke': 'Why did the football player bring a pencil to the game?\n\nBecause he wanted to *draw* a foul!', 'explanation': 'This joke is a pun that plays on the two different meanings of the word "draw":\n\n1.  **"Draw" (with a pencil):** This is the literal meaning implied by the football player bringing a pencil to the game. You use a pencil to create a picture or sketch. The setup makes you think of artistic drawing.\n\n2.  **"Draw a foul" (in sports):** In football (or soccer, basketball, etc.), to "draw a foul" means to provoke or cause an opposing player to commit an infraction against you. It\'s about getting the referee to call a penalty on the other team.\n\nThe humor comes from the **unexpected switch** in the meaning of "draw." The setup leads you to think of the artistic meaning, but the punchline uses the sports-specific meaning, creating a silly and clever play on words.'}, next=(), config={'configurable': {'thread_id': '2', 'chec

#### Updating State

In [32]:
workflow.update_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f177cd5-6d1d-667a-8000-b94df2510676", "checkpoint_ns": ""}}, {'topic':'Tenis'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f177e31-edab-6f9e-8001-e8c23360e6c0'}}

In [33]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Tenis'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f177e31-edab-6f9e-8001-e8c23360e6c0'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-07-04T20:01:18.412995+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f177cd5-6d1d-667a-8000-b94df2510676'}}, tasks=(PregelTask(id='6e2c1a46-7677-6342-4ba4-18d79aa8cded', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'football', 'joke': 'Why did the football player bring a pencil to the game?\n\nBecause he wanted to *draw* a foul!', 'explanation': 'This joke is a pun that plays on the two different meanings of the word "draw":\n\n1.  **"Draw" (with a pencil):** This is the literal meaning implied by the football player bringing a pencil to the game. You use a 

### Fault Tolerance

In [18]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [19]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [20]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(30)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [21]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [22]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
✅ Step 3 executed


In [23]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...

✅ Final State: {'input': 'start', 'step1': 'done', 'step2': 'done'}


In [24]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f177deb-71a2-6315-8003-d489fba53dcb'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-07-04T19:29:46.358453+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f177deb-7194-678c-8002-a2c3b91291be'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f177deb-7194-678c-8002-a2c3b91291be'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-04T19:29:46.352833+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f177dea-5360-6135-8001-c8f58bac3024'}}, tasks=(PregelTask(id='a3050909-3110-3ccd-df37-6378a784367